# **Semantic Web & Linked Open Data with Python**
## บทที่ 06 — SPARQL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/semantic-web/blob/main/code/ch06/ch06_notebook.ipynb)

> ## *Author* : [Rathachai CHAWUTHAI](https://rathachai.creatier.pro/) , Ph.D
> ### *Affiliation* : Computer Engineering, King Mongkut's Institute of Technology Ladkrabang (KMITL)
> #### *Updated Date* : 2026-08-23
---

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work (course text) is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Sample code cells are additionally released under the MIT License.

**References:**
*   https://www.w3.org/TR/sparql11-query/
*   https://query.wikidata.org/

📖 เอกสารประกอบบทนี้: [`06-sparql/`](https://github.com/Rathachai/semantic-web/tree/main/06-sparql)

---

### Setup

In [1]:
!pip install -q rdflib

In [2]:
from rdflib import Graph, Namespace
UNI = Namespace("http://example.org/university#")

data_ttl = """
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix uni:  <http://example.org/university#> .

uni:alice a uni:Student ; uni:name "Alice" ; uni:gpa 3.8 ; uni:dept uni:cs .
uni:bob   a uni:Student ; uni:name "Bob"   ; uni:gpa 2.1 ; uni:dept uni:cs .
uni:carol a uni:Student ; uni:name "Carol" ; uni:gpa 3.2 ; uni:dept uni:math .
uni:dave  a uni:Student ; uni:name "Dave"  ; uni:gpa 2.9 ; uni:dept uni:math .
uni:cs   uni:name "Computer Science" .
uni:math uni:name "Mathematics" .
uni:alice uni:advisorOf uni:bob .
uni:bob   uni:advisorOf uni:carol .
"""
g = Graph().parse(data=data_ttl, format="turtle")
print(f"{len(g)} triples")

20 triples


## SELECT พื้นฐาน

In [3]:
res = g.query("""
    SELECT ?name ?gpa WHERE {
      ?s a uni:Student ; uni:name ?name ; uni:gpa ?gpa .
    } ORDER BY DESC(?gpa)
""", initNs={"uni": UNI})
for row in res:
    print(row.name, row.gpa)

Alice 3.8
Carol 3.2
Dave 2.9
Bob 2.1


## FILTER

In [4]:
res = g.query("""
    SELECT ?name ?gpa WHERE {
      ?s uni:name ?name ; uni:gpa ?gpa .
      FILTER(?gpa >= 3.0)
    }
""", initNs={"uni": UNI})
for row in res:
    print(row.name, row.gpa)

Alice 3.8
Carol 3.2


## OPTIONAL

In [5]:
res = g.query("""
    SELECT ?name ?advisee WHERE {
      ?s uni:name ?name .
      OPTIONAL { ?s uni:advisorOf ?adv . ?adv uni:name ?advisee . }
    }
""", initNs={"uni": UNI})
for row in res:
    print(row.name, "->", row.advisee)

Alice -> Bob
Bob -> Carol
Carol -> None
Dave -> None
Computer Science -> None
Mathematics -> None


## Aggregate: GROUP BY + AVG

In [6]:
res = g.query("""
    SELECT ?deptName (AVG(?gpa) AS ?avgGpa) (COUNT(?s) AS ?n) WHERE {
      ?s uni:gpa ?gpa ; uni:dept ?dept .
      ?dept uni:name ?deptName .
    } GROUP BY ?deptName
""", initNs={"uni": UNI})
for row in res:
    print(f"{row.deptName:20s} avg={float(row.avgGpa):.2f}  n={row.n}")

Computer Science     avg=2.95  n=2
Mathematics          avg=3.05  n=2


## Property Path: หา advisor-chain ทั้งหมด (0 ขั้นขึ้นไป)

In [7]:
res = g.query("""
    SELECT ?descendant WHERE {
      uni:alice uni:advisorOf+ ?descendant .
    }
""", initNs={"uni": UNI})
for row in res:
    print(row.descendant)

http://example.org/university#bob
http://example.org/university#carol


## CONSTRUCT: สร้างกราฟใหม่

In [8]:
new_g = g.query("""
    CONSTRUCT { ?s uni:isHighAchiever true . }
    WHERE { ?s uni:gpa ?gpa . FILTER(?gpa > 3.0) }
""", initNs={"uni": UNI}).graph
print(new_g.serialize(format="turtle"))

@prefix ns1: <http://example.org/university#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ns1:alice ns1:isHighAchiever true .

ns1:carol ns1:isHighAchiever true .




## ASK

In [9]:
res = g.query("ASK { uni:alice uni:advisorOf uni:bob . }", initNs={"uni": UNI})
print(bool(res))

True


## 🎯 สรุป

- **SELECT** คืนตาราง, **CONSTRUCT** คืนกราฟใหม่, **ASK** คืน boolean
- **OPTIONAL** เติมค่าถ้ามี ไม่ตัดแถวถ้าไม่มี
- **Aggregate ต้องมากับ GROUP BY** เสมอ
- **Property path** (`+`, `*`, `?`) เดินตามความสัมพันธ์หลายขั้นในคำสั่งเดียว

ลองเปลี่ยน `?gpa >= 3.0` เป็นเงื่อนไขอื่น หรือเพิ่ม `advisorOf` เป็นทอดที่ 3 แล้วดูผล property path!

---

## จบบทที่ 06 🎉

ไปต่อบทถัดไป: **บทที่ 07 — Python Semantic Web**

📚 กลับหน้าคอร์ส: [https://github.com/Rathachai/semantic-web](https://github.com/Rathachai/semantic-web)

---
https://github.com/Rathachai/semantic-web

*Semantic Web & Linked Open Data with Python*